In [17]:
# SPDX-License-Identifier: CC-BY-4.0
# Code for "Active Continual Learning with Metaplastic Binary Bayesian Neural Networks"
# Kellian Cottart, Théo Ballet, Djohan Bonnet, Damien Querlioz
# Portions of the code are adapted from the Pytorch project (BSD-3-Clause)
# Author: Kellian Cottart <kellian.cottart@gmail.com>
# Date: 2025-30-01

In [18]:

import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import os
import seaborn as sns
import re
import json
import pandas as pd
AXESSIZE = 28
FONTSIZE = 26
TICKSIZE = 24   
LEGENDSIZE = 26
plt.rcParams['svg.fonttype'] = 'none'
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.05"
results_folder = "results-main-openloris"
df = pd.DataFrame()
# iterate through all root folders in the results folder
for folder in os.listdir(results_folder):
    current_path = os.path.join(results_folder, folder)
    # extract the name from the first config
    config_path = os.path.join(current_path, "config0/config.json")
    with open(config_path, "r") as f:
        config = json.load(f)
    # n_iterations is the number of config folders
    n_iterations = len([f for f in os.listdir(current_path) if f.startswith("config") and os.path.isdir(os.path.join(current_path, f))])
    
    # Add the row of parameters to the dataframe   
    field = [key for key in config.keys() if "ewc" in key]
    field_si = [key for key in config.keys() if key=="si"]
    row = {
        "path": current_path,
        "opt": config["optimizer"],
        "n_tasks": config["n_tasks"],
        "n_epochs": config["epochs"],
        "n_train_samples": config["n_train_samples"] if "n_train_samples" in config else 1,
        "n_test_samples": config["n_test_samples"] if "n_test_samples" in config else 1,
        "n_iterations": n_iterations,
        "batch_size": config["train_batch_size"],
        "method": list(config["network_params"]["active_learning"].keys())[0] if "active_learning" in config["network_params"] else "none",
        "N": str(config["optimizer_params"]["N"]) if "N" in config["optimizer_params"] else "none",
        # key containing ewc but not strictly equal to ewc
        "ewc": config[field[0]] if len(field) > 0 else "none",
        "si": config[field_si[0]] if len(field_si) > 0 else "none",
        "subfeatures": int(config["task_params"]["subfeatures"]) if "subfeatures" in config["task_params"] else 25088,
    }
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
# One color for each path
colors = sns.color_palette("viridis", len(df["method"].unique()))
markers = ["D", "o", "s", "h", "^", "x", "v", "p", "*", "X", "D", "o", "s", "h", "^", "x", "v", "p", "*", "X"]

In [19]:
data = []
for idx, row in df.iterrows():
    path = row["path"]
    n_tasks = row["n_tasks"]
    n_epochs = row["n_epochs"]
    n_iterations = row["n_iterations"]
    full_accuracies = []
    full_roc_aucs = []
    for it in range(n_iterations):
        current_it_path = os.path.join(path, f"config{it}")
        accuracy_path = os.path.join(current_it_path, "accuracy")
        uncertainty_path = os.path.join(current_it_path, "uncertainty")
        accuracies = []
        roc_aucs = []
        for task in range(n_tasks):
            for epoch in range(n_epochs):
                suffix = f"task={task}-epoch={epoch}.npy"
                accuracies.append(jnp.load(os.path.join(accuracy_path, "split=0-"+suffix)))
                aleatoric_u = jnp.load(os.path.join(uncertainty_path, f"roc-auc-aleatoric-{suffix}"))
                epistemic_u = jnp.load(os.path.join(uncertainty_path, f"roc-auc-epistemic-{suffix}"))
                variation_ratio_u = jnp.load(os.path.join(uncertainty_path, f"roc-auc-variation-ratio-{suffix}"))
                roc_aucs.append((aleatoric_u, epistemic_u, variation_ratio_u))

        full_accuracies.append(jnp.array(accuracies))
        full_roc_aucs.append(jnp.array(roc_aucs))
    full_accuracies = jnp.array(full_accuracies)*100
    accuracy_array_mean = jnp.mean(full_accuracies, -1)[:, -1].mean(0)
    accuracy_array_std = jnp.mean(full_accuracies, -1)[:, -1].std(0)
    full_roc_aucs = jnp.array(full_roc_aucs)
    data.append((accuracy_array_mean, accuracy_array_std, full_accuracies[:, -1, :], full_roc_aucs, full_accuracies))
# Add new columns to df
df["accuracies_mean"] = [d[0] for d in data]
df["accuracies_std"] = [d[1] for d in data]
df["last_accuracies"] = [d[2] for d in data]
df["roc_aucs"] = [d[3] for d in data]
df["accuracies"] = [d[4] for d in data]

In [20]:
import pandas as pd
import jax.numpy as jnp

def make_key(row):
    key = row["opt"]
    if row["ewc"] != "none":
        key += "-ewc"
    if row["si"] != "none":
        key += "-si"
    return key


def compute_accuracy_stats(last_accuracies):
    mean_all = jnp.mean(jnp.mean(last_accuracies, axis=-1))
    std_all = jnp.std(jnp.mean(last_accuracies, axis=-1))
    return f"${mean_all:.2f} \\pm {std_all:.2f}$"


def compute_bwt(accuracies):
    acc = accuracies.mean(0) / 100
    n_tasks = acc.shape[1]
    bwt = jnp.mean(
        jnp.array([acc[-1, t] - acc[t, t] for t in range(n_tasks - 1)])
    )
    return f"{bwt:.4f}"


def compute_roc_auc_stats(roc_aucs, idx):
    values = roc_aucs[:, :, idx].mean(-1)
    mean = values.mean(0)
    std = values.std(0)
    return f"${mean:.4f} \\pm {std:.4f}$"


def build_tables_for_subfeatures(df_sub):
    rows = {}

    for _, row in df_sub.iterrows():
        key = make_key(row)

        rows[key] = {
            "Accuracy (%)": compute_accuracy_stats(row["last_accuracies"]),
            "BWT": compute_bwt(row["accuracies"]),
            "ROC-AUC Aleatoric": compute_roc_auc_stats(row["roc_aucs"], 0),
            "ROC-AUC Epistemic": compute_roc_auc_stats(row["roc_aucs"], 1),
            "ROC-AUC Variation Ratio": compute_roc_auc_stats(row["roc_aucs"], 2),
        }

    table = pd.DataFrame.from_dict(rows, orient="index")

    # pretty names
    table.index = (
        table.index
        .str.replace("bayesbinn", "Bayes BiNN")
        .str.replace("bimu", "BiMU")
        .str.replace("mesu", "MESU")
        .str.replace("adam", "STE")
        .str.replace("sgd-ewc", "EWC Online")
        .str.replace("sgd-si", "Synaptic Intelligence")
        .str.replace("sgd", "SGD")
        .str.replace("synapticmetaplasticity", "Synaptic Metaplasticity")
    )
    return table

tables_per_subfeatures = {}
for subf, df_sub in df.groupby("subfeatures"):
    tables_per_subfeatures[subf] = build_tables_for_subfeatures(df_sub)
    
final_table = pd.concat(
    tables_per_subfeatures,
    names=["Subfeatures", "Method"]
)

In [21]:
final_table

Accuracy (%)      BWT  \
Subfeatures Method                                               
1024        Bayes BiNN               $72.01 \pm 1.69$  -0.1366   
            BiMU                     $73.61 \pm 1.53$  -0.1561   
            MESU                     $80.82 \pm 0.94$  -0.1137   
            EWC Online               $75.72 \pm 0.87$  -0.1006   
            SGD                      $61.28 \pm 1.25$  -0.1966   
            Synaptic Intelligence    $70.75 \pm 0.57$  -0.0561   
            STE                      $52.88 \pm 3.39$  -0.2259   
            Synaptic Metaplasticity  $62.82 \pm 2.31$  -0.1410   
8192        Bayes BiNN               $86.93 \pm 0.41$  -0.0682   
            BiMU                     $89.19 \pm 0.19$  -0.0917   
            MESU                     $87.01 \pm 0.69$  -0.0575   
            EWC Online               $87.18 \pm 0.58$  -0.0498   
            STE                      $79.12 \pm 1.39$  -0.1131   
            Synaptic Metaplasticity  $88.03 \pm 0.38$  -0.0408   
            SGD                      $86.27 \pm 0.51$  -0.0646   
            Synaptic Intelligence    $86.44 \pm 0.66$  -0.0326   
25088       BiMU                     $90.62 \pm 0.22$  -0.0525   
            MESU                     $87.84 \pm 0.11$  -0.0515   
            SGD                      $88.04 \pm 0.03$  -0.0517   
            Synaptic Intelligence    $88.04 \pm 0.03$  -0.0517   
            STE                      $83.79 \pm 1.13$  -0.0790   
            Synaptic Metaplasticity  $86.72 \pm 0.34$  -0.0485   
            EWC Online               $88.23 \pm 0.05$  -0.0452   
            Bayes BiNN               $89.37 \pm 0.77$  -0.0399   

                                       ROC-AUC Aleatoric    ROC-AUC Epistemic  \
Subfeatures Method                                                              
1024        Bayes BiNN               $0.9321 \pm 0.0112$  $0.9982 \pm 0.0002$   
            BiMU                     $0.9579 \pm 0.0112$  $0.9987 \pm 0.0001$   
            MESU                     $0.8958 \pm 0.0128$  $0.9928 \pm 0.0016$   
            EWC Online               $0.9658 \pm 0.0146$  $0.0000 \pm 0.0000$   
            SGD                      $0.0000 \pm 0.0000$  $0.0000 \pm 0.0000$   
            Synaptic Intelligence    $0.9509 \pm 0.0259$  $0.0000 \pm 0.0000$   
            STE                      $0.7331 \pm 0.0156$  $0.0000 \pm 0.0000$   
            Synaptic Metaplasticity  $0.7222 \pm 0.0401$  $0.0000 \pm 0.0000$   
8192        Bayes BiNN               $0.9868 \pm 0.0012$  $0.9990 \pm 0.0000$   
            BiMU                     $0.9897 \pm 0.0045$  $0.9990 \pm 0.0000$   
            MESU                     $0.9980 \pm 0.0009$  $0.9767 \pm 0.0030$   
            EWC Online               $0.9982 \pm 0.0012$  $0.0000 \pm 0.0000$   
            STE                      $0.6138 \pm 0.0458$  $0.0000 \pm 0.0000$   
            Synaptic Metaplasticity  $0.6335 \pm 0.0346$  $0.0000 \pm 0.0000$   
            SGD                      $0.9977 \pm 0.0015$  $0.0000 \pm 0.0000$   
            Synaptic Intelligence    $0.9982 \pm 0.0009$  $0.0000 \pm 0.0000$   
25088       BiMU                     $0.9254 \pm 0.0012$  $0.8969 \pm 0.0037$   
            MESU                     $0.8323 \pm 0.0021$  $0.8556 \pm 0.0023$   
            SGD                      $0.8293 \pm 0.0006$  $0.0000 \pm 0.0000$   
            Synaptic Intelligence    $0.8293 \pm 0.0006$  $0.0000 \pm 0.0000$   
            STE                      $0.5531 \pm 0.0045$  $0.0000 \pm 0.0000$   
            Synaptic Metaplasticity  $0.5513 \pm 0.0046$  $0.0000 \pm 0.0000$   
            EWC Online               $0.7939 \pm 0.0009$  $0.0000 \pm 0.0000$   
            Bayes BiNN               $0.9198 \pm 0.0035$  $0.8965 \pm 0.0050$   

                                    ROC-AUC Variation Ratio  
Subfeatures Method                                           
1024        Bayes BiNN                  $0.9604 \pm 0.0062$  
            BiMU               